# 05 · Multiple datasets — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1. Relative weighting

Inflating InSAR sigma lowers its precision quadratically and moves the joint
solution toward the GNSS-only estimate.

## 2–3. A second track and sign error

Look vectors with different horizontal signs separate east from vertical.
Reversing one look vector makes coherent, sign-reversed residuals.

## 4. Explicit stacking

Concatenated observations and block weights must use the same dataset order as
the vertically stacked Green's matrices.


In [ ]:
look_e, look_n, look_u = 0.4, -0.1, 0.91
look_norm = np.sqrt(look_e**2 + look_n**2 + look_u**2)
look_e, look_n, look_u = look_e / look_norm, look_n / look_norm, look_u / look_norm
los = east * look_e + north * look_n + up * look_u
insar = geodef.data.insar(
    lon=station_lon, lat=station_lat, los=los,
    sigma=np.full(los.size, sigma),
    look_e=look_e, look_n=look_n, look_u=look_u,
    name="ascending",
)
joint = geodef.solve(
    fault, datasets=[gnss, insar], components="dip",
    regularization="laplacian", regularization_strength=1.0,
)
diagnostics = geodef.invert.diagnostics(joint)
stacked = geodef.greens.stack_obs([gnss, insar])
assert stacked.size == gnss.n_obs + insar.n_obs
print({name: value.reduced_chi2 for name, value in diagnostics.items()})


## Interpretation and alternatives

Dataset-specific diagnostics are the fastest way to see whether density or reported uncertainty has created imbalance.
